# **Maestría en Inteligencia Artificial Aplicada**

## Curso: **Análisis de grandes volumenes de datos**

### Tecnológico de Monterrey

### Prof Dr. Iván Olmos Pineda

## Actividad Semana 8

### **Métricas de calidad de resultados**

#### **Nombre y matrícula**

*   Emmanuel Merida Toledo A01795858


In [111]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [112]:
# imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum
from pyspark.sql.functions import round, when
from pyspark.sql.functions import hour, dayofweek

from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler

from pyspark.ml.classification import LogisticRegression
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.clustering import KMeans

from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.evaluation import ClusteringEvaluator


In [113]:
spark = SparkSession.builder.getOrCreate()

# **Punto - 1:** Construcción de la muestra M

In [114]:
# Ruta
ruta_archivo = "/content/drive/MyDrive/MNA/Analisis de grandes volumenes de datos/Data/particiones"

# Cargar cada partición
p1 = spark.read.csv(f"{ruta_archivo}/output_ETHBTC_M5.csv/", header=True, inferSchema=True)
p2 = spark.read.csv(f"{ruta_archivo}/output_BTCUSDT_M15.csv/", header=True, inferSchema=True)
p3 = spark.read.csv(f"{ruta_archivo}/output_BNBUSDT_H1.csv/", header=True, inferSchema=True)
p4 = spark.read.csv(f"{ruta_archivo}/output_ETHUSDT_M30.csv/", header=True, inferSchema=True)

In [115]:
# Aplicar muestreo aleatorio simple del 10% en cada partición
m1 = p1.sample(withReplacement=False, fraction=0.1, seed=42)
m2 = p2.sample(withReplacement=False, fraction=0.1, seed=42)
m3 = p3.sample(withReplacement=False, fraction=0.1, seed=42)
m4 = p4.sample(withReplacement=False, fraction=0.1, seed=42)

df_M = m1.union(m2).union(m3).union(m4)

In [116]:
print("Total de registros en M:", df_M.count())
df_M.printSchema()

Total de registros en M: 109743
root
 |-- datetime: timestamp (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- close: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- coin: string (nullable = true)
 |-- frequency: string (nullable = true)



In [117]:
# Validación de nulos
df_M.select([sum(col(c).isNull().cast("int")).alias(c) for c in df_M.columns]).show()

+--------+----+----+---+-----+------+----+---------+
|datetime|open|high|low|close|volume|coin|frequency|
+--------+----+----+---+-----+------+----+---------+
|       0|   0|   0|  0|    0|     0|   0|        0|
+--------+----+----+---+-----+------+----+---------+



In [118]:
# Se agrega columna que servira como criterio
df_M = df_M.withColumn("label", when(col("close") > col("open"), 1).otherwise(0))

# Verificacion de balanceo para el criterio definido
df_M.groupBy("label").count() \
     .withColumn("porcentaje", round(col("count") / df_M.count() * 100, 2)) \
     .orderBy("label") \
     .show()

+-----+-----+----------+
|label|count|porcentaje|
+-----+-----+----------+
|    0|55985|     51.01|
|    1|53758|     48.99|
+-----+-----+----------+



# **Punto - 2:** Construcción Train – Test

In [119]:
# División 70/30 por cada muestra Mᵢ (m1 a m4)
spark.conf.set("spark.sql.shuffle.partitions", "200")
tr1, ts1 = m1.randomSplit([0.7, 0.3], seed=42)
tr2, ts2 = m2.randomSplit([0.7, 0.3], seed=42)
tr3, ts3 = m3.randomSplit([0.7, 0.3], seed=42)
tr4, ts4 = m4.randomSplit([0.7, 0.3], seed=42)

# Unión total de entrenamiento y prueba
train_df = tr1.union(tr2).union(tr3).union(tr4)
test_df = ts1.union(ts2).union(ts3).union(ts4)

# Verificación de tamaños
print("Tamaño conjunto de entrenamiento:", train_df.count())
print("Tamaño conjunto de prueba:", test_df.count())

Tamaño conjunto de entrenamiento: 77367
Tamaño conjunto de prueba: 32376


In [120]:
# Transformar el datetime
train_df = train_df.withColumn("hour", hour("datetime")) \
                   .withColumn("day_of_week", dayofweek("datetime"))

# Indexar coin y frequency
indexer_coin = StringIndexer(inputCol="coin", outputCol="coin_index").fit(train_df)
indexer_freq = StringIndexer(inputCol="frequency", outputCol="freq_index").fit(train_df)

train_df = indexer_coin.transform(train_df)
train_df = indexer_freq.transform(train_df)

# Vector assembler
assembler = VectorAssembler(
    inputCols=["open", "high", "low", "close", "volume", "hour", "day_of_week", "coin_index", "freq_index"],
    outputCol="features"
)
train_df = assembler.transform(train_df)

# Aplicar exactamente los mismos pasos al test_df
test_df = test_df.withColumn("hour", hour("datetime")) \
                 .withColumn("day_of_week", dayofweek("datetime"))

test_df = indexer_coin.transform(test_df)
test_df = indexer_freq.transform(test_df)
test_df = assembler.transform(test_df)

# Se agrega columna que servira como criterio
train_df = train_df.withColumn("label", when(col("close") > col("open"), 1).otherwise(0))
test_df  = test_df.withColumn("label", when(col("close") > col("open"), 1).otherwise(0))

In [121]:
# Verificación de balance en ambos conjuntos
for subset, nombre in zip([train_df, test_df], ["Entrenamiento", "Prueba"]):
    print(f"Distribución en {nombre}:")
    subset.groupBy("label").count() \
          .withColumn("porcentaje", round(col("count") / subset.count() * 100, 2)) \
          .orderBy("label") \
          .show()


Distribución en Entrenamiento:
+-----+-----+----------+
|label|count|porcentaje|
+-----+-----+----------+
|    0|39466|     51.01|
|    1|37901|     48.99|
+-----+-----+----------+

Distribución en Prueba:
+-----+-----+----------+
|label|count|porcentaje|
+-----+-----+----------+
|    0|16519|     51.02|
|    1|15857|     48.98|
+-----+-----+----------+



Se realizó una división del 70% para entrenamiento y 30% para prueba, respetando las particiones originales `Mᵢ` (`m1`, `m2`, `m3`, `m4`) generadas a partir de las distintas criptomonedas del corpus. La división se realizó por separado sobre cada una de las particiones utilizando la función `randomSplit()` de PySpark, y posteriormente se unieron todos los subconjuntos resultantes.

Se ajusto el numero de particiones a 200 por la cantidad de registros la sumatoria de todas las muestras.

La unión final generó dos conjuntos globales:
- `train_df`: conjunto de entrenamiento
- `test_df`: conjunto de prueba

El preprocesamiento se aplicó después de la división, garantizando que las transformaciones no aprendieran información del conjunto de prueba. Las transformaciones aplicadas fueron:

- Extracción de variables temporales `hour` y `day_of_week` desde `datetime`.
- Codificación de las variables categóricas `coin` y `frequency` mediante `StringIndexer`, ajustadas (`fit`) sobre `train_df` y luego aplicadas (`transform`) en ambos conjuntos.
- Combinación de todas las variables numéricas y categóricas en una única columna `features` utilizando `VectorAssembler`.

También se añadió la columna `label`, que indica si el precio de cierre (`close`) fue mayor al de apertura (`open`). Esta columna se utiliza como variable objetivo para los modelos supervisados.

Finalmente, se verificó que la distribución de clases en ambos conjuntos fuera balanceada:

- **Entrenamiento**:
  - `label = 0`: 51.01%
  - `label = 1`: 48.99%
  
- **Prueba**:
  - `label = 0`: 51.02%
  - `label = 1`: 48.98%

Dado que la distribución natural de la variable `label` ya era balanceada, **no fue necesario aplicar técnicas de muestreo estratificado**. El uso de `randomSplit()` resultó suficiente para mantener proporciones consistentes entre clases sin introducir sesgos.


# **Punto - 3:** Selección de métricas para medir calidad de resultados

Para evaluar los modelos entrenados, se seleccionaron métricas proporcionadas por la clase `MulticlassClassificationEvaluator` de PySpark, que permiten obtener una visión completa del desempeño global del modelo.

Las métricas utilizadas son:

- **Accuracy**: mide el porcentaje total de predicciones correctas sobre el total de ejemplos.
- **F1-score ponderado**: combina precisión y recall en una única métrica, promediada según la frecuencia de cada clase.
- **Precisión ponderada**: mide cuántas de las predicciones positivas fueron realmente positivas, considerando todas las clases.
- **Recall ponderado**: mide cuántos de los ejemplos positivos reales fueron correctamente identificados, también considerando todas las clases.

Estas métricas están diseñadas para entornos multiclase, pero son igualmente aplicables en clasificación binaria, ya que entregan una evaluación global del modelo considerando el equilibrio entre clases y la capacidad general de predicción.


### Modelos no supervisados
En caso de aplicar técnicas como `KMeans`, se empleará:

- **Silhouette Score**: mide qué tan similar es un punto a su propio clúster en comparación con otros clústeres.

Esta métrica es escalable y está optimizada para grandes volúmenes de datos, por lo que es compatible con la naturaleza de esta actividad.


# **Punto - 4:** Entrenamiento de Modelos de Aprendizaje

**Aprendizaje Supervisado — Regresión Logística**

Se utilizó un modelo de **regresión logística sin regularización** (`regParam=0.0`) con un máximo de 100 iteraciones. Este modelo fue entrenado sobre el conjunto `train_df` y aplicado sobre `test_df`.

In [122]:
# Entrenar modelo de regresión logística
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    regParam=0.0,
)

modelo_lr = lr.fit(train_df)
predicciones = modelo_lr.transform(test_df)

**Aprendizaje Supervisado — GBTClassifier**

También se entrenó un modelo de tipo **Gradient Boosted Trees** (`GBTClassifier`) con los siguientes hiperparámetros:

- maxIter=100: número de iteraciones del boost
- maxDepth=8: profundidad máxima de cada árbol
- stepSize=0.2: tasa de aprendizaje

In [123]:
# Entrenar modelo
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=100,
    maxDepth=8,
    stepSize=0.2)

modelo_gbt = gbt.fit(train_df)
predicciones_gbt = modelo_gbt.transform(test_df)

**Aprendizaje no supervisado — KMeans**

Se aplicó el algoritmo **KMeans** con `k=3` para identificar agrupamientos naturales en los datos sin utilizar la variable objetivo. El modelo se entrenó sobre los mismos datos de entrenamiento (`train_df`) y luego se aplicó al conjunto de prueba (`test_df`).

In [124]:
# Crear y entrenar el modelo KMeans
kmeans = KMeans(featuresCol="features", predictionCol="cluster", k=3, seed=42)
modelo_kmeans = kmeans.fit(train_df)

df_clusterizado = modelo_kmeans.transform(test_df)

# **Punto - 5:** Análisis de resultados

 **Aprendizaje Supervisado**

 Se compararon dos modelos: **GBTClassifier** (árboles potenciados por gradiente) y **Regresión Logística**. A continuación, se muestran sus métricas clave, calculadas usando `MulticlassClassificationEvaluator` de PySpark:

In [125]:
metricas = {
    "accuracy",
    "f1",
    "weightedPrecision",
    "weightedRecall"
}

**Metricas GBTClassifier**

In [126]:
predicciones_gbt.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0| 9217|
|    0|       1.0| 7302|
|    1|       0.0| 8423|
|    1|       1.0| 7434|
+-----+----------+-----+



In [127]:
print("Metricas GBTClassifier")
for metric in metricas:
    evaluador = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric
    )
    resultado = evaluador.evaluate(predicciones_gbt)
    print(f"{metric}: {resultado:.4f}")


Metricas GBTClassifier
weightedRecall: 0.5143
f1: 0.5134
weightedPrecision: 0.5137
accuracy: 0.5143


**Metricas Regresión Logística**

In [128]:
print("Regresión Logística")
for metric in metricas:
    evaluador = MulticlassClassificationEvaluator(
        labelCol="label",
        predictionCol="prediction",
        metricName=metric
    )
    resultado = evaluador.evaluate(predicciones)
    print(f"{metric}: {resultado:.4f}")

Regresión Logística
weightedRecall: 0.6885
f1: 0.6528
weightedPrecision: 0.8042
accuracy: 0.6885


In [129]:
predicciones.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|16489|
|    0|       1.0|   30|
|    1|       0.0|10054|
|    1|       1.0| 5803|
+-----+----------+-----+



El modelo de **Regresión Logística** superó claramente al GBTClassifier en todas las métricas evaluadas. Su precisión ponderada de 0.80 demuestra que realizó predicciones positivas con alto nivel de acierto, y su F1-score indica un buen equilibrio entre precisión y recall. Por tanto, se considera el modelo con mejor desempeño en este contexto.

**Aprendizaje No Supervisado**

Para el modelo no supervisado, se utilizó el algoritmo **KMeans** con `k=3` y se evaluó la calidad del agrupamiento utilizando la métrica **Silhouette Score**.

Este valor indica una muy buena cohesión dentro de los clústeres y una separación clara entre ellos. El resultado sugiere que el modelo KMeans logró identificar agrupamientos significativos en los datos, incluso sin conocer previamente la variable objetivo.

**Metricas KMeans**

In [130]:
# Evaluar con Silhouette Score
evaluador = ClusteringEvaluator(
    predictionCol="cluster",
    featuresCol="features",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

silhouette_score = evaluador.evaluate(df_clusterizado)
print(f"Silhouette Score: {silhouette_score:.4f}")

Silhouette Score: 0.9121


**Conclusión General**

- **Regresión Logística** resultó ser el modelo supervisado más efectivo, combinando alta precisión y recall.
- El **modelo GBTClassifier** mostró un desempeño inferior, probablemente por no captar patrones lineales tan bien como el modelo lineal.
- El modelo **KMeans**, evaluado con Silhouette Score, demostró una excelente estructura de clústeres, útil para exploración y segmentación.

